# 01 — Exploratory Data Analysis & Preprocessing

**Owner**: Noah  
**Course**: Machine Learning III (Unsupervised Learning) — Albert School  
**Dataset**: AI4I 2020 Predictive Maintenance (10 000 observations)

**Goal**: understand the data, identify cleaning needs, and design the preprocessing pipeline that will be consumed by the four anomaly detection models (Isolation Forest, One-Class SVM, LOF, Elliptic Envelope).

**Key constraint**: the column `Machine failure` (and its subtypes `TWF`, `HDF`, `PWF`, `OSF`, `RNF`) must NOT be used as a training signal. We only inspect the failure rate to inform the `contamination` hyperparameter, then set the labels aside for the Part 4 final evaluation.

## 1. Imports and configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = Path.cwd().parent / "ai4i2020.csv"

## 2. Load the dataset

We load the CSV and look at the first rows to confirm the schema.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Schema and missing values

We expect 14 columns: 2 identifiers (`UDI`, `Product ID`), 1 categorical (`Type`), 5 numeric sensors, and 6 label columns (the global `Machine failure` flag plus 5 failure subtypes).

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

**Observation**: 10 000 rows × 14 columns, **zero missing values**. This is a clean industrial dataset, so we can skip imputation entirely. All numeric features are already in the right dtype (`float64` for temperatures and torque, `int64` for rotational speed and tool wear).

In [ ]:
df.describe()

## 4. Identify columns to keep, drop, and hold out

For unsupervised anomaly detection we must isolate three groups:

- **Identifiers** (`UDI`, `Product ID`): drop — no information for an anomaly model.
- **Held-out labels** (`Machine failure` + 5 failure subtypes `TWF`, `HDF`, `PWF`, `OSF`, `RNF`): drop from training. The 5 subtypes are essentially leaks of `Machine failure`. We keep `Machine failure` aside for the Part 4 "reveal".
- **Features** (`Type` + 5 numeric sensors): used for training.

In [ ]:
ID_COLUMNS = ["UDI", "Product ID"]
LABEL_COLUMNS = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
NUMERIC_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
CATEGORICAL_FEATURES = ["Type"]

y_true = df["Machine failure"].copy()
X = df.drop(columns=ID_COLUMNS + LABEL_COLUMNS)

print(f"X shape: {X.shape}")
print(f"X columns: {list(X.columns)}")
print(f"y_true held out: {y_true.shape[0]} rows")